Part 1: Environment Setup and Path Configuration

In [6]:
import pandas as pd
import numpy as np
import ast
import os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
import sklearn_crfsuite
from nltk.tokenize.treebank import TreebankWordDetokenizer

BASE_PATH = Path(r'C:\Users\Lenovo\Desktop\NLP project\ebm_nlp_2_00')
INPUT_CSV = BASE_PATH / "axis2_B" / "outputs" / "sentence_records_df.csv"
EMBEDDINGS_PATH = BASE_PATH / "axis2_B" / "data" / "pubmedbert_embeddings.npy"
GOLD_DIR = BASE_PATH / "annotations" / "aggregated" / "hierarchical_labels" / "participants" / "test" / "gold"
OUTPUT_DIR = BASE_PATH / "axis2_B" / "outputs"

def safe_eval(val):
    if isinstance(val, list): return val
    try: return ast.literal_eval(val)
    except: return []

all_df = pd.read_csv(INPUT_CSV)
all_df['sentence_tokens'] = all_df['sentence_tokens'].apply(safe_eval)

sentence_embeddings = np.load(EMBEDDINGS_PATH)

official_test_ids = [f.split('.')[0] for f in os.listdir(GOLD_DIR) if f.endswith('.ann')]
official_test_ids = [int(i) for i in official_test_ids]

test_df = all_df[all_df['doc_id'].isin(official_test_ids)].copy()
train_df = all_df[~all_df['doc_id'].isin(official_test_ids)].copy()

X_test_embeddings = sentence_embeddings[test_df.index]
X_train_embeddings = sentence_embeddings[train_df.index]

print(f"Total Sentences: {len(all_df)}")
print(f"Official Test Docs Found: {len(test_df['doc_id'].unique())}")
print(f"Training Sentences: {len(train_df)}")
print(f"Testing Sentences: {len(test_df)}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Lenovo\\Desktop\\NLP project\\ebm_nlp_2_00\\axis2_B\\data\\pubmedbert_embeddings.npy'

Step 2: Sentence Embedding (PubMedBERT)

In [3]:
embed_model = SentenceTransformer('microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext')

X_train_embeddings = embed_model.encode(train_df['sentence_text'].tolist(), show_progress_bar=True)
X_test_embeddings = embed_model.encode(test_df['sentence_text'].tolist(), show_progress_bar=True)

No sentence-transformers model found with name microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1497 [00:00<?, ?it/s]

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Step 3: CRF Feature Engineering

In [8]:
def word2features(sent_tokens, i):
    word = str(sent_tokens[i])
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word.isupper()': word.isupper(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        features['-1:word.lower()'] = str(sent_tokens[i-1]).lower()
        if i > 1:
            features['-2:word.lower()'] = str(sent_tokens[i-2]).lower()
    else:
        features['BOS'] = True
    if i < len(sent_tokens) - 1:
        features['+1:word.lower()'] = str(sent_tokens[i+1]).lower()
        if i < len(sent_tokens) - 2:
            features['+2:word.lower()'] = str(sent_tokens[i+2]).lower()
    else:
        features['EOS'] = True
    return features

import numpy as np
import os
from pathlib import Path

sentence_embeddings = np.concatenate([X_train_embeddings, X_test_embeddings])

EMBEDDINGS_PATH = Path(r'C:\Users\Lenovo\Desktop\NLP project\ebm_nlp_2_00\axis2_B\data\pubmedbert_embeddings.npy')
EMBEDDINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(EMBEDDINGS_PATH, sentence_embeddings)

GOLD_DIR = Path(r'C:\Users\Lenovo\Desktop\NLP project\ebm_nlp_2_00\annotations\aggregated\hierarchical_labels\participants\test\gold')
official_test_ids = [int(f.split('.')[0]) for f in os.listdir(GOLD_DIR) if f.endswith('.ann')]

test_df = all_df[all_df['doc_id'].isin(official_test_ids)].copy()
train_df = all_df[~all_df['doc_id'].isin(official_test_ids)].copy()

X_test_embeddings = sentence_embeddings[test_df.index]
X_train_embeddings = sentence_embeddings[train_df.index]

print(f"Test docs matched: {len(test_df['doc_id'].unique())}")

Test docs matched: 184


Step 4: Hybrid Model Training and Prediction

In [ ]:
import os
import pandas as pd
import numpy as np
import nltk
import re
import sklearn_crfsuite
from sklearn.ensemble import RandomForestClassifier
from nltk.tokenize.treebank import TreebankWordDetokenizer
from pathlib import Path

def word2features_v19(sent, i, pos):
    w = str(sent[i])
    t = pos[i][1]
    med_suffixes = ('ine', 'one', 'ide', 'ate', 'pam', 'lol', 'vir', 'mab', 'tin', 'mic', 'cin')
    
    f = {
        'bias': 1.0,
        'w.lower()': w.lower(),
        'w.isupper()': w.isupper(),
        'w.isdigit()': w.isdigit(), 
        'w.med_suffix': w.lower().endswith(med_suffixes),
        'pos': t,
        'pos2': t[:2]
    }
    if i > 0:
        f.update({'-1:w.lower()': str(sent[i-1]).lower(), '-1:pos': pos[i-1][1]})
    else:
        f['BOS'] = True
    if i < len(sent) - 1:
        f.update({'+1:w.lower()': str(sent[i+1]).lower(), '+1:pos': pos[i+1][1]})
    else:
        f['EOS'] = True
    return f

def synthesize_labels_v19(pos, t_type):
    anchors = {
        'P': {'patients', 'men', 'women', 'adults', 'children', 'subjects', 'volunteers', 'infants', 'elderly', 'cases', 'smokers'},
        'I': {'mg', 'dose', 'treatment', 'therapy', 'tablet', 'drug', 'injection', 'infusion', 'placebo', 'capsule', 'intervention'},
        'O': {'rate', 'score', 'ratio', 'survival', 'incidence', 'primary', 'secondary', 'outcome', 'change', 'mortality', 'improvement'}
    }
    target = anchors[t_type]
    labels = ['O'] * len(pos)
    
    for i, (w, t) in enumerate(pos):
        if w.lower() in target:
            for j in range(max(0, i-2), min(len(pos), i+3)):
                if pos[j][1].startswith(('NN', 'JJ', 'CD')):
                    labels[j] = 'I'
    return labels

def clean_v19(entities, t_type):
    hdr = r'^(CONCLUSION|METHODS|RESULTS|BACKGROUND|DESIGN|OBJECTIVE|SETTING|AIM|POPULATION|STATISTICAL|STUDY)\s*[:\-]?\s*'
    frg = {'of', 'and', 'the', 'next', 'more', 'than', 'need', 'for', 'with', 'between', 'after', 'from', 'at', 'in', 'to', 'on'}
    
    res = []
    detok = TreebankWordDetokenizer()
    for e in entities:
        e = re.sub(hdr, '', e, flags=re.IGNORECASE).strip()
        tokens = e.split()
        if len(tokens) < 1 or len(tokens) > 5 or e.isupper(): continue
        if tokens[0].lower() in frg or tokens[-1].lower() in frg: continue
        
        f = re.sub(r'^(the|a|an|with|for|of|in|at|to|on|and)\s+', '', e, flags=re.IGNORECASE).strip()
        if len(f) > 2: res.append(f)
            
    final = []
    for x in sorted(list(set(res)), key=len, reverse=True):
        if not any(x.lower() in u.lower() and x.lower() != u.lower() for u in final):
            final.append(x)
    return "; ".join(final[:3]) 

detok = TreebankWordDetokenizer()
test_ids = test_df['doc_id'].unique()
final_output = pd.DataFrame({'doc_id': [str(i) for i in test_ids]})

if 'pos_tags' not in train_df.columns: train_df['pos_tags'] = train_df['sentence_tokens'].apply(nltk.pos_tag)
if 'pos_tags' not in test_df.columns: test_df['pos_tags'] = test_df['sentence_tokens'].apply(nltk.pos_tag)

configs = [
    {'col': 'p_count', 'target': 'participants_pred', 'type': 'P', 'rf_lim': 0.40},
    {'col': 'i_count', 'target': 'interventions_pred', 'type': 'I', 'rf_lim': 0.50},
    {'col': 'o_count', 'target': 'outcomes_pred', 'type': 'O', 'rf_lim': 0.50}
]

for cfg in configs:
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42).fit(X_train_embeddings, (train_df[cfg['col']] > 0).astype(int))
    sub = train_df[train_df[cfg['col']] > 0]
    crf = sklearn_crfsuite.CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=60).fit(
        [[word2features_v19(r['sentence_tokens'], j, r['pos_tags']) for j in range(len(r['sentence_tokens']))] for _, r in sub.iterrows()],
        [synthesize_labels_v19(r['pos_tags'], cfg['type']) for _, r in sub.iterrows()])
    
    test_df['prob'] = rf.predict_proba(X_test_embeddings)[:, 1]
    col_results = []
    for d_id, gp in test_df.groupby('doc_id'):
        raw = []
        for _, row in gp[gp['prob'] >= cfg['rf_lim']].nlargest(3, 'prob').iterrows():
            preds = crf.predict_single([word2features_v19(row['sentence_tokens'], j, row['pos_tags']) for j in range(len(row['sentence_tokens']))])
            curr = []
            for word, label in zip(row['sentence_tokens'], preds):
                if label == 'I': curr.append(word)
                elif curr: raw.append(detok.detokenize(curr)); curr = []
            if curr: raw.append(detok.detokenize(curr))
        col_results.append({'doc_id': str(d_id), cfg['target']: clean_v19(raw, cfg['type'])})
    
    final_output = final_output.merge(pd.DataFrame(col_results), on='doc_id', how='left')

final_output.fillna("").to_csv(OUTPUT_DIR / "Axis2B_RESULTS2.csv", index=False)